# Phase 6 — Hyperparameter Tuning and Final Model Evaluation

This notebook tunes the strongest baseline models identified during Phase 5, compares the tuned configurations, selects the final classifier, evaluates it on the untouched test dataset, and prepares the fitted pipeline for deployment.

## Objectives

- Reproduce the exact training, validation, and test splits used during baseline modelling.
- Preserve the test dataset until final model selection.
- Tune the strongest baseline models using stratified cross-validation.
- Use Macro F1 as the primary optimisation metric.
- Compare tuned models against their baseline counterparts.
- Select and refit the final machine-learning pipeline.
- Perform one final evaluation using the untouched test set.
- Save the fitted preprocessing and prediction pipeline for application use.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
current_path = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [
    current_path,
    *current_path.parents,
]:
    preprocessing_path = (
        candidate
        / "src"
        / "preprocessing.py"
    )

    if preprocessing_path.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root"
    )

print(
    "Project root:",
    PROJECT_ROOT,
)

Project root: C:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System


In [3]:
project_root_string = str(
    PROJECT_ROOT
)

if project_root_string not in sys.path:
    sys.path.insert(
        0,
        project_root_string,
    )

In [4]:
from src.preprocessing import (
    PREDICTIVE_FEATURES,
    build_preprocessor,
)

In [5]:
print(
    "Number of predictive features:",
    len(PREDICTIVE_FEATURES),
)

print(
    "Predictive features:"
)

for feature in PREDICTIVE_FEATURES:
    print("-", feature)

Number of predictive features: 16
Predictive features:
- Age
- Height
- Weight
- FCVC
- NCP
- CH2O
- FAF
- TUE
- CAEC
- CALC
- Gender
- family_history_with_overweight
- FAVC
- SMOKE
- SCC
- MTRANS


In [6]:
preprocessor_check = build_preprocessor()

print(
    "Preprocessor type:",
    type(preprocessor_check).__name__,
)

Preprocessor type: ColumnTransformer


In [7]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

BASELINE_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "generated"
    / "baseline_model_comparison.csv"
)

TUNING_CANDIDATES_PATH = (
    PROJECT_ROOT
    / "reports"
    / "generated"
    / "tuning_candidates.csv"
)

print("Dataset:", DATA_PATH)
print("Baseline report:", BASELINE_REPORT_PATH)
print(
    "Tuning candidates:",
    TUNING_CANDIDATES_PATH,
)

Dataset: C:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System\data\raw\obesity.csv
Baseline report: C:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System\reports\generated\baseline_model_comparison.csv
Tuning candidates: C:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System\reports\generated\tuning_candidates.csv


In [8]:
assert DATA_PATH.exists(), (
    f"Dataset not found: {DATA_PATH}"
)

assert BASELINE_REPORT_PATH.exists(), (
    "Baseline comparison report was not found"
)

assert TUNING_CANDIDATES_PATH.exists(), (
    "Tuning candidates report was not found"
)

print(
    "All required Phase 6 input files exist"
)

All required Phase 6 input files exist


In [9]:
IDENTIFIER_COLUMN = "id"
TARGET_COLUMN = "NObeyesdad"
RANDOM_STATE = 42

In [10]:
CLASS_ORDER = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

In [11]:
df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape,
)

df.head()

Dataset shape: (20758, 18)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [12]:
print(
    "Number of rows:",
    len(df),
)

print(
    "Number of columns:",
    len(df.columns),
)

print(
    "Target classes:",
    df[TARGET_COLUMN].nunique(),
)

Number of rows: 20758
Number of columns: 18
Target classes: 7


In [13]:
X = df[
    PREDICTIVE_FEATURES
].copy()

y = df[
    TARGET_COLUMN
].copy()

print(
    "Feature matrix shape:",
    X.shape,
)

print(
    "Target shape:",
    y.shape,
)

Feature matrix shape: (20758, 16)
Target shape: (20758,)


In [14]:
assert list(
    X.columns
) == PREDICTIVE_FEATURES

assert len(
    X.columns
) == 16

assert len(
    X
) == len(
    y
)

print(
    "Feature configuration verified"
)

Feature configuration verified


In [15]:
X_train, X_temp, y_train, y_temp = (
    train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
)

In [16]:
X_validation, X_test, y_validation, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )
)

In [17]:
split_summary_df = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Validation",
            "Test",
        ],
        "Records": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            len(X_train) / len(X) * 100,
            len(X_validation) / len(X) * 100,
            len(X_test) / len(X) * 100,
        ],
    }
)

split_summary_df.round(2)

,Dataset,Records,Percentage
0,Training,14530,70.0
1,Validation,3114,15.0
2,Test,3114,15.0


In [18]:
train_validation_overlap = (
    X_train.index
    .intersection(
        X_validation.index
    )
)

train_test_overlap = (
    X_train.index
    .intersection(
        X_test.index
    )
)

validation_test_overlap = (
    X_validation.index
    .intersection(
        X_test.index
    )
)

print(
    "Train-validation overlap:",
    len(train_validation_overlap),
)

print(
    "Train-test overlap:",
    len(train_test_overlap),
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap),
)

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [19]:
class_distribution_df = pd.DataFrame(
    {
        "Full Dataset": (
            y.value_counts(
                normalize=True
            )
        ),
        "Training": (
            y_train.value_counts(
                normalize=True
            )
        ),
        "Validation": (
            y_validation.value_counts(
                normalize=True
            )
        ),
        "Test": (
            y_test.value_counts(
                normalize=True
            )
        ),
    }
).reindex(CLASS_ORDER)

class_distribution_df.round(4)

,Full Dataset,Training,Validation,Test
NObeyesdad,,,,
Insufficient_Weight,0.1215,0.1215,0.1217,0.1214
Normal_Weight,0.1485,0.1485,0.1487,0.1484
Overweight_Level_I,0.1169,0.1169,0.1169,0.1169
Overweight_Level_II,0.1215,0.1215,0.1214,0.1217
Obesity_Type_I,0.1402,0.1402,0.1400,0.1403
Obesity_Type_II,0.1565,0.1565,0.1564,0.1564
Obesity_Type_III,0.1949,0.1949,0.1949,0.1949


In [20]:
baseline_results_df = pd.read_csv(
    BASELINE_REPORT_PATH
)

baseline_results_df

,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1,Macro F1 Rank,Balanced Accuracy Rank,Accuracy Rank
0,Gradient Boosting,0.903340,0.893142,0.893031,0.902952,1,1,1
1,Random Forest,0.895633,0.884543,0.884858,0.895254,2,2,2
2,Logistic Regression,0.858060,0.843242,0.841809,0.856599,3,3,3
3,Decision Tree,0.845215,0.831168,0.830801,0.845645,4,4,4
4,Dummy Classifier,0.194926,0.142857,0.046608,0.063596,5,5,5


In [21]:
tuning_candidates_df = pd.read_csv(
    TUNING_CANDIDATES_PATH
)

tuning_candidates_df

,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1,Training Macro F1,Macro F1 Gap
0,Gradient Boosting,0.903340,0.893142,0.893031,0.902952,0.912625,0.019594
1,Random Forest,0.895633,0.884543,0.884858,0.895254,1.000000,0.115142


In [22]:
tuning_candidate_names = (
    tuning_candidates_df[
        "Model"
    ]
    .tolist()
)

print(
    "Models selected for tuning:"
)

for model_number, model_name in enumerate(
    tuning_candidate_names,
    start=1,
):
    print(
        f"{model_number}. {model_name}"
    )

Models selected for tuning:
1. Gradient Boosting
2. Random Forest


In [23]:
assert len(
    tuning_candidate_names
) == 2

assert set(
    tuning_candidate_names
) == {
    "Gradient Boosting",
    "Random Forest",
}

print(
    "Phase 5 tuning candidates verified"
)

Phase 5 tuning candidates verified


In [24]:
baseline_results_lookup = (
    baseline_results_df
    .set_index("Model")
)

gradient_boosting_baseline_macro_f1 = (
    baseline_results_lookup.loc[
        "Gradient Boosting",
        "Macro F1",
    ]
)

random_forest_baseline_macro_f1 = (
    baseline_results_lookup.loc[
        "Random Forest",
        "Macro F1",
    ]
)

print(
    "Gradient Boosting baseline Macro F1:",
    round(
        gradient_boosting_baseline_macro_f1,
        4,
    ),
)

print(
    "Random Forest baseline Macro F1:",
    round(
        random_forest_baseline_macro_f1,
        4,
    ),
)

Gradient Boosting baseline Macro F1: 0.893
Random Forest baseline Macro F1: 0.8849


In [25]:
development_summary = {
    "Total records": len(X),
    "Training records": len(X_train),
    "Validation records": len(X_validation),
    "Reserved test records": len(X_test),
    "Predictive features": len(PREDICTIVE_FEATURES),
    "Target classes": y.nunique(),
    "Tuning candidates": len(
        tuning_candidate_names
    ),
}

pd.Series(
    development_summary,
    name="Value",
)

Total records            20758
Training records         14530
Validation records        3114
Reserved test records     3114
Predictive features         16
Target classes               7
Tuning candidates            2
Name: Value, dtype: int64

In [26]:
assert df.shape == (
    20758,
    18,
)

assert X.shape == (
    20758,
    16,
)

assert len(y) == 20758

assert len(X_train) == 14530
assert len(X_validation) == 3114
assert len(X_test) == 3114

assert len(y_train) == 14530
assert len(y_validation) == 3114
assert len(y_test) == 3114

assert len(
    train_validation_overlap
) == 0

assert len(
    train_test_overlap
) == 0

assert len(
    validation_test_overlap
) == 0

assert y.nunique() == 7

assert len(
    tuning_candidate_names
) == 2

assert set(
    tuning_candidate_names
) == {
    "Gradient Boosting",
    "Random Forest",
}